# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors Exploration with `mlcroissant`

This notebook provides a guided workflow for loading, exploring, and processing the [FAIR^2 dataset package](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json) using the `mlcroissant` library, following the Croissant schema specification.

### Dataset Source
The dataset is defined by a Croissant schema file accessible at:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`


In [ ]:
# Ensure `mlcroissant` and required libraries are installed
!pip install mlcroissant pandas matplotlib

## 1. Data Loading
Load the dataset and its metadata using `mlcroissant`. This will allow exploration of the schema, record sets, fields, and subsequent record access.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset (schema and metadata)
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset: {metadata.name}")
print(f"Description: {metadata.description}")

## 2. Data Overview
List all available record sets and their fields, referencing them by their unique `@id`. This helps to understand the schema structure and know which fields to work with.

In [ ]:
# List all record sets and their fields by @id
print("Available record sets and fields:")
record_sets = list(dataset.list_record_sets())  # returns list of (record_set_id, RecordSet)
for record_set_id, record_set_obj in record_sets:
    print(f"\nRecord Set @id: {record_set_id}")
    fields = record_set_obj.fields
    if not fields:
        print("  (No fields defined)")
    else:
        for field_id, field in fields.items():
            print(f"  Field @id: {field_id}, name: {getattr(field, 'name', '[no name]')}, type: {getattr(field, 'data_type', '[unknown type]')}")
    print("  (To access records use this record set @id)")

## 3. Data Extraction
We extract tabular data for analysis. Here, we'll load all records from the main (likely only) record set into a DataFrame for exploration.

Make sure to reference the record set and fields by their `@id` as shown above.

In [ ]:
# Select record set(s) to extract (replace with record set @id from overview step)
# Use the first available record set if only one exists
record_sets_ids = [rs_id for rs_id, _ in record_sets]
dataframes = {}

for record_set_id in record_sets_ids:
    # Fetch records from each record set
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} rows from record set {record_set_id}")
    print(f"Columns: {list(df.columns)}\n")

# Pick the first record set for subsequent analysis
main_record_set_id = record_sets_ids[0]
df = dataframes[main_record_set_id]
df.head()

## 4. Exploratory Data Analysis (EDA)
Let's process and analyze the tabular data. We'll demonstrate filtering by a numeric field, normalization, and simple group-by analysis referencing fields by their `@id`. Adjust field IDs to your interests as found in overview.

In [ ]:
# Identify numeric fields to use (choose by @id)
# For illustration, let's try 'Age_at_CRCDiagnosis' or similar. Adjust the field ID as needed.
# If unsure, print(df.dtypes) to inspect numeric fields.

print("Available DataFrame columns (field @id):", list(df.columns))
numeric_candidate_fields = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
print("Numeric field candidates:", numeric_candidate_fields)

# Example: Assume field '@id' for age is 'age_at_crc' or similar -- adjust as per actual field
numeric_field = numeric_candidate_fields[0] if numeric_candidate_fields else None

if numeric_field:
    # Example filtering and normalizing
    threshold = df[numeric_field].quantile(0.25)  # Use first quartile as threshold example
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    print(filtered_df[[numeric_field]].head())

    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Identify categorical/text columns for group-by
    group_field = None
    for col in df.columns:
        if df[col].dtype == 'object' and col != numeric_field:
            group_field = col
            break

    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame(name=f"mean_{numeric_field}")
        print(f"\nGrouped data by {group_field} (mean {numeric_field}):")
        print(grouped_df.head())
else:
    print("No numeric field found for EDA.")

## 5. Visualization
Visualize the distribution of a numeric field and its relationship to a categorical variable (if available).

In [ ]:
# Simple histogram and boxplot visualization
if numeric_field:
    plt.figure(figsize=(10, 4))
    plt.subplot(1,2,1)
    df[numeric_field].hist(bins=10, color='skyblue')
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')

    if group_field:
        plt.subplot(1,2,2)
        df.boxplot(column=numeric_field, by=group_field, grid=False)
        plt.title(f'{numeric_field} by {group_field}')
        plt.xlabel(group_field)
        plt.suptitle("")

    plt.tight_layout()
    plt.show()
else:
    print("No numeric field found for plotting.")

## 6. Conclusion
In this notebook, we accessed structured clinical and molecular data for second primary colorectal cancer survivors using `mlcroissant`, explored its schema, extracted the records, and performed basic data analysis and visualization. You can adapt field and record set `@id`s in this workflow to suit any FAIR Croissant dataset for reproducible, schema-driven exploration.